# Hindi UPSC Essay Corpus — One-Click Research Run

This notebook is self-contained: the five source JSON files are embedded in the notebook, so you do **not** need to upload separate files. In Google Colab, choose **Runtime → Run all**. It writes a ZIP of tables, predictions, plots, and run metadata, then offers the ZIP for download.

**Scope and cautions**
- Uses the five supplied sources: Human, GPT, Gemini, Grok, and Sarvam.
- Removes duplicate `(source, record ID)` rows, keeping the first record. Grok has 103 rows but 102 unique IDs.
- Groups cross-validation by normalized canonical Human topic title to reduce same-topic leakage; duplicate titles are kept in the same group.
- Reports character n-gram and word n-gram TF-IDF + LinearSVC baselines. Scores describe this dataset and are not proof of reliable general-purpose AI detection.
- Similarity scores are descriptive lexical-overlap measures, not semantic similarity or plagiarism scores.
- This notebook does not include the DeepSeek holdout or external UD-HDTB/PUD data.


In [1]:
# 1. Imports and embedded source data
import os, json, gzip, base64, re, unicodedata, zipfile, warnings, io, textwrap
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix
)

# Embedded compressed JSON data; no separate uploads required.
DATA_B64 = (
"..."
)
raw_sources = json.loads(gzip.decompress(base64.b64decode(DATA_B64)).decode("utf-8"))
print("Embedded sources loaded:", {k: len(v) for k, v in raw_sources.items()})


Embedded sources loaded: {'Gemini': 102, 'GPT': 102, 'Grok': 103, 'Human': 102, 'Sarvam': 102}


In [2]:
# 2. Normalize records, validate all five sources, and audit duplicates
EXPECTED = ["Human", "GPT", "Gemini", "Grok", "Sarvam"]

def extract_text(rec):
    for key in ("content", "text", "essay", "output", "response"):
        value = rec.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return ""

def normalize_title(value):
    value = unicodedata.normalize("NFKC", str(value or "")).lower()
    value = re.sub(r"[\W_]+", " ", value, flags=re.UNICODE)
    return " ".join(value.split())

human_by_id = {}
for r in raw_sources["Human"]:
    rid = str(r.get("id") or r.get("topic") or "").strip()
    if rid and rid not in human_by_id:
        human_by_id[rid] = str(r.get("title") or r.get("theme") or "").strip()

records = []
audit_rows = []
for source in EXPECTED:
    seen = set()
    for i, r in enumerate(raw_sources.get(source, [])):
        rid = str(r.get("id") or r.get("topic") or f"row_{i}").strip()
        txt = extract_text(r)
        title = str(r.get("title") or r.get("theme") or "").strip()
        duplicate = rid in seen
        seen.add(rid)
        audit_rows.append({
            "source": source, "row_number": i, "record_id": rid,
            "title": title, "has_text": bool(txt), "word_count": len(txt.split()),
            "duplicate_source_id": duplicate
        })
        if not txt or duplicate:
            continue
        canonical_title = human_by_id.get(rid, title)
        records.append({
            "source": source, "record_id": rid,
            "title": title, "canonical_title": canonical_title,
            "prompt_group": normalize_title(canonical_title),
            "text": txt, "word_count": len(txt.split())
        })

audit_df = pd.DataFrame(audit_rows)
df = pd.DataFrame(records)
missing = [s for s in EXPECTED if s not in set(df["source"])]
if missing:
    raise ValueError("Required source(s) missing or no essay text extracted: " + ", ".join(missing))
source_counts = df.groupby("source").size().to_dict()
if set(source_counts) != set(EXPECTED):
    raise ValueError(f"Expected all five sources, got: {source_counts}")

print("Rows after duplicate-ID removal:", len(df))
print("Rows by source:", source_counts)
print("Unique prompt groups:", df["prompt_group"].nunique())
print("Duplicate input rows:")
print(audit_df.groupby("source").agg(input_rows=("record_id","size"),
      unique_ids=("record_id","nunique"), empty_text_rows=("has_text",lambda x: int((~x).sum())),
      duplicate_rows=("duplicate_source_id","sum")).to_string())

# Keep an explicit audit table for repeated normalized titles.
topic_audit = (df.groupby(["prompt_group","canonical_title"], dropna=False)
               .agg(records=("record_id","size"), sources=("source","nunique"),
                    record_ids=("record_id", lambda x: ";".join(sorted(set(x)))))
               .reset_index())


Rows after duplicate-ID removal: 510
Rows by source: {'GPT': 102, 'Gemini': 102, 'Grok': 102, 'Human': 102, 'Sarvam': 102}
Unique prompt groups: 100
Duplicate input rows:
        input_rows  unique_ids  empty_text_rows  duplicate_rows
source                                                         
GPT            102         102                0               0
Gemini         102         102                0               0
Grok           103         102                0               1
Human          102         102                0               0
Sarvam         102         102                0               0


In [3]:
# 3. Grouped cross-validation helpers and evaluation
# Each topic group is kept entirely within one fold to reduce topic leakage.
def grouped_oof_predictions(data, text_col, label_col, group_col, mode, n_splits=5):
    y = data[label_col].astype(str).to_numpy()
    groups = data[group_col].astype(str).to_numpy()
    unique_groups = len(set(groups))
    splits = min(n_splits, unique_groups)
    if splits < 2:
        raise ValueError("Need at least two distinct topic groups for grouped CV.")
    splitter = StratifiedGroupKFold(n_splits=splits, shuffle=True, random_state=42)
    pred = np.empty(len(data), dtype=object)
    fold_rows = []
    for fold, (train_idx, test_idx) in enumerate(splitter.split(data[text_col], y, groups), start=1):
        if mode == "char":
            vec = TfidfVectorizer(analyzer="char", ngram_range=(2,5),
                                  min_df=2, max_features=250000, sublinear_tf=True)
        elif mode == "word":
            vec = TfidfVectorizer(analyzer="word", ngram_range=(1,2),
                                  token_pattern=r"(?u)\b\w+\b", min_df=2,
                                  max_features=250000, sublinear_tf=True)
        else:
            raise ValueError(mode)
        X_train = vec.fit_transform(data.iloc[train_idx][text_col].astype(str))
        X_test = vec.transform(data.iloc[test_idx][text_col].astype(str))
        clf = LinearSVC(class_weight="balanced", random_state=42)
        clf.fit(X_train, y[train_idx])
        fold_pred = clf.predict(X_test)
        pred[test_idx] = fold_pred
        fold_rows.append({
            "fold": fold,
            "train_rows": len(train_idx),
            "test_rows": len(test_idx),
            "test_groups": len(set(groups[test_idx])),
            "accuracy": accuracy_score(y[test_idx], fold_pred),
            "balanced_accuracy": balanced_accuracy_score(y[test_idx], fold_pred),
            "macro_f1": f1_score(y[test_idx], fold_pred, average="macro", zero_division=0)
        })
    if pd.isna(pred).any():
        raise ValueError("Some records did not receive an out-of-fold prediction.")
    return pred, pd.DataFrame(fold_rows)

def summarize_predictions(y_true, y_pred, task, mode):
    labels = sorted(set(y_true))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    report = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
    row = {
        "task": task, "features": mode,
        "n": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0)
    }
    return row, pd.DataFrame(cm, index=labels, columns=labels), pd.DataFrame(report).T

results = []
predictions = {}
fold_tables = {}
confusion_tables = {}
report_tables = {}

# Five-way source attribution: source labels are the target classes.
for mode in ("char", "word"):
    pred, folds = grouped_oof_predictions(df, "text", "source", "prompt_group", mode)
    key = f"five_way_{mode}"
    predictions[key] = pd.DataFrame({
        "record_id": df["record_id"], "source_true": df["source"],
        "source_predicted": pred, "canonical_title": df["canonical_title"],
        "prompt_group": df["prompt_group"], "word_count": df["word_count"]
    })
    fold_tables[key] = folds
    row, cm, rep = summarize_predictions(df["source"].to_numpy(), pred, "five_way_source", mode)
    results.append(row); confusion_tables[key] = cm; report_tables[key] = rep

# Binary Human-vs-AI detection.
binary_df = df.copy()
binary_df["label_binary"] = np.where(binary_df["source"].eq("Human"), "Human", "AI")
for mode in ("char", "word"):
    pred, folds = grouped_oof_predictions(binary_df, "text", "label_binary", "prompt_group", mode)
    key = f"binary_{mode}"
    predictions[key] = pd.DataFrame({
        "record_id": binary_df["record_id"], "source": binary_df["source"],
        "label_true": binary_df["label_binary"], "label_predicted": pred,
        "canonical_title": binary_df["canonical_title"],
        "prompt_group": binary_df["prompt_group"], "word_count": binary_df["word_count"]
    })
    fold_tables[key] = folds
    row, cm, rep = summarize_predictions(binary_df["label_binary"].to_numpy(), pred, "binary_human_vs_ai", mode)
    human_mask = binary_df["label_binary"].eq("Human").to_numpy()
    row["human_false_positive_rate"] = float(np.mean(np.asarray(pred)[human_mask] == "AI"))
    row["human_false_positives"] = int(np.sum(np.asarray(pred)[human_mask] == "AI"))
    row["human_count"] = int(human_mask.sum())
    results.append(row); confusion_tables[key] = cm; report_tables[key] = rep

summary_df = pd.DataFrame(results)
print(summary_df.to_string(index=False))


              task features   n  accuracy  balanced_accuracy  macro_f1  human_false_positive_rate  human_false_positives  human_count
   five_way_source     char 510  0.998039           0.998039  0.998039                        NaN                    NaN          NaN
   five_way_source     word 510  0.996078           0.996078  0.996069                        NaN                    NaN          NaN
binary_human_vs_ai     char 510  0.994118           0.985294  0.990706                   0.029412                    3.0        102.0
binary_human_vs_ai     word 510  0.992157           0.980392  0.987561                   0.039216                    4.0        102.0


In [4]:
# 4. Descriptive lexical similarity and length summaries
# Similarity is descriptive only; all essays are used to construct this shared TF-IDF space.
# It is not a held-out predictive evaluation and is not semantic similarity.
sim_vectorizer = TfidfVectorizer(analyzer="word", ngram_range=(1,2),
                                 token_pattern=r"(?u)\b\w+\b", min_df=2,
                                 max_features=250000, sublinear_tf=True)
X_all = sim_vectorizer.fit_transform(df["text"].astype(str))
row_positions = {idx: pos for pos, idx in enumerate(df.index)}
human_df = df[df["source"].eq("Human")].copy()
sim_rows = []
for _, r in df[df["source"].ne("Human")].iterrows():
    match = human_df[human_df["record_id"].eq(r["record_id"])]
    if match.empty:
        continue
    hidx = match.index[0]
    a_pos = row_positions[r.name]
    h_pos = row_positions[hidx]
    cosine = float(X_all[a_pos].multiply(X_all[h_pos]).sum())
    sim_rows.append({
        "record_id": r["record_id"], "canonical_title": r["canonical_title"],
        "source": r["source"], "human_word_count": int(match.iloc[0]["word_count"]),
        "ai_word_count": int(r["word_count"]), "tfidf_cosine": cosine
    })
similarity_df = pd.DataFrame(sim_rows)
similarity_summary = (similarity_df.groupby("source")
                      .agg(n=("tfidf_cosine","size"),
                           mean_tfidf_cosine=("tfidf_cosine","mean"),
                           median_tfidf_cosine=("tfidf_cosine","median"),
                           sd_tfidf_cosine=("tfidf_cosine","std"),
                           mean_human_words=("human_word_count","mean"),
                           mean_source_words=("ai_word_count","mean"))
                      .reset_index())
length_summary = (df.groupby("source")
                  .agg(n=("word_count","size"), mean_words=("word_count","mean"),
                       median_words=("word_count","median"), sd_words=("word_count","std"),
                       min_words=("word_count","min"), max_words=("word_count","max"))
                  .reset_index())
print("Similarity summary:")
print(similarity_summary.to_string(index=False))
print("\nLength summary:")
print(length_summary.to_string(index=False))


Similarity summary:
source   n  mean_tfidf_cosine  median_tfidf_cosine  sd_tfidf_cosine  mean_human_words  mean_source_words
   GPT 102           0.313444             0.317070         0.049503       1458.990196         988.333333
Gemini 102           0.294285             0.288731         0.037266       1458.990196         995.813725
  Grok 102           0.311963             0.306969         0.044454       1458.990196         551.343137
Sarvam 102           0.306267             0.307716         0.038418       1458.990196        1190.137255

Length summary:
source   n  mean_words  median_words   sd_words  min_words  max_words
   GPT 102  988.333333         985.0  87.647622        807       1232
Gemini 102  995.813725         999.0  75.047144        827       1190
  Grok 102  551.343137         510.5 144.993581        370       1232
 Human 102 1458.990196        1404.0 462.077048        354       2686
Sarvam 102 1190.137255        1172.5 122.006252       1008       1592


In [5]:
# 5. Export every result and package it for download
OUT = Path("results_hindi_upsc_oneclick")
OUT.mkdir(exist_ok=True)

audit_df.to_csv(OUT / "source_row_audit.csv", index=False, encoding="utf-8-sig")
df.drop(columns=["text"]).to_csv(OUT / "cleaned_corpus_metadata.csv", index=False, encoding="utf-8-sig")
topic_audit.to_csv(OUT / "topic_group_audit.csv", index=False, encoding="utf-8-sig")
summary_df.to_csv(OUT / "evaluation_summary.csv", index=False, encoding="utf-8-sig")
similarity_df.to_csv(OUT / "paired_topic_tfidf_similarity.csv", index=False, encoding="utf-8-sig")
similarity_summary.to_csv(OUT / "tfidf_similarity_summary.csv", index=False, encoding="utf-8-sig")
length_summary.to_csv(OUT / "word_count_summary.csv", index=False, encoding="utf-8-sig")

for key, table in predictions.items():
    table.to_csv(OUT / f"predictions_{key}.csv", index=False, encoding="utf-8-sig")
for key, table in fold_tables.items():
    table.to_csv(OUT / f"fold_metrics_{key}.csv", index=False, encoding="utf-8-sig")
for key, table in confusion_tables.items():
    table.to_csv(OUT / f"confusion_{key}.csv", encoding="utf-8-sig")
for key, table in report_tables.items():
    table.to_csv(OUT / f"classification_report_{key}.csv", encoding="utf-8-sig")

# Figures: confusion matrices and word-count distributions.
for key, cm in confusion_tables.items():
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm.values, aspect="auto")
    ax.set_xticks(range(len(cm.columns))); ax.set_xticklabels(cm.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(cm.index))); ax.set_yticklabels(cm.index)
    ax.set_xlabel("Predicted label"); ax.set_ylabel("True label"); ax.set_title(f"Confusion matrix: {key}")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm.iloc[i,j]), ha="center", va="center")
    fig.tight_layout(); fig.savefig(OUT / f"confusion_{key}.png", dpi=160); plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
for source in EXPECTED:
    vals = df.loc[df["source"].eq(source), "word_count"]
    ax.hist(vals, bins=20, alpha=0.45, label=source)
ax.set_xlabel("Whitespace-separated word count"); ax.set_ylabel("Number of essays")
ax.set_title("Essay length by source"); ax.legend(); fig.tight_layout()
fig.savefig(OUT / "essay_length_by_source.png", dpi=160); plt.close(fig)

metadata = {
    "sources_expected": EXPECTED,
    "source_counts_after_deduplication": source_counts,
    "input_rows": {k: len(v) for k,v in raw_sources.items()},
    "unique_topic_groups": int(df["prompt_group"].nunique()),
    "duplicate_policy": "Keep first row per (source, record_id); repeated normalized canonical Human titles are grouped together.",
    "evaluation": "5-fold StratifiedGroupKFold where groups are normalized canonical Human topic titles; character 2-5 grams and word 1-2 grams; TF-IDF + LinearSVC.",
    "cautions": [
        "Scores apply only to this corpus and setup; do not interpret as general-purpose detector performance.",
        "Source attribution can exploit source-specific formatting or genre cues.",
        "TF-IDF cosine is lexical overlap, not semantic similarity or plagiarism.",
        "DeepSeek holdout and external human corpora are not included."
    ]
}
(OUT / "run_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

zip_name = "results_hindi_upsc_oneclick.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, arcname=str(p.relative_to(OUT)))

print("Saved output folder:", OUT.resolve())
print("Created results ZIP:", Path(zip_name).resolve(), "| bytes:", Path(zip_name).stat().st_size)

# Download automatically in Google Colab; harmless if not running in Colab.
try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print("Not running in Google Colab. Download the ZIP from the notebook working directory.")


Saved output folder: C:\Users\Harsh Singh Parmar\Documents\Projects\Hindi Test\results_hindi_upsc_oneclick
Created results ZIP: C:\Users\Harsh Singh Parmar\Documents\Projects\Hindi Test\results_hindi_upsc_oneclick.zip | bytes: 278016
Not running in Google Colab. Download the ZIP from the notebook working directory.
